# Worked Example: Feedback-Locked Epochs with Behavior Metadata

## Goal
Epoch around real `feedback_start` times and attach reward/RPE for condition contrasts.


In [ ]:
import pandas as pd
from pathlib import Path
from LFPAnalysis import build_event_locked_pipeline_config, run_pipeline

beh = pd.read_csv(Path('../../data/sample_beh.csv'))
config = build_event_locked_pipeline_config(
    Path('../../data/sample_ieeg_bp.fif'),
    file_format='mne',
    event_name='feedback_start',
    event_times=beh['feedback_start'].tolist(),
    baseline_mode='zscore',
    baseline_window=(-0.5, 0.0),
    tmin=-0.5,
    tmax=1.5,
    metadata={'reward': beh['reward'].tolist(), 'rpe': beh['rpe'].tolist()},
)
result = run_pipeline(config)
epochs = result.epochs
print(f'{len(epochs)} epochs, reward={epochs.metadata.reward.sum():.0f} win / {(epochs.metadata.reward==0).sum():.0f} loss')
print(result.baseline_summary.head())

## Plot evoked reward vs no-reward (ACC channel)

In [ ]:
import matplotlib.pyplot as plt

chan = 'racas1-racas2'
reward_evoked = epochs[epochs.metadata['reward'] == 1].copy().pick([chan]).average()
loss_evoked = epochs[epochs.metadata['reward'] == 0].copy().pick([chan]).average()
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(reward_evoked.times, reward_evoked.data[0], label='reward')
ax.plot(loss_evoked.times, loss_evoked.data[0], label='no reward')
ax.axvline(0, color='k', ls='--', lw=0.8)
ax.set(xlabel='Time (s)', ylabel='Amplitude (a.u.)', title=f'Evoked {chan}')
ax.legend()
fig.tight_layout()
plt.show()

## Next step

Chapter 08 (`08_first_psd_and_fooof`) covers PSD and FOOOF.